# Next-Word Prediction v2 — LSTM vs GRU on Shakespeare

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adhiraj170204/Next-Word-Prediction/blob/main/notebooks/next_word_prediction.ipynb)

This notebook trains a word-level next-word predictor, the same kind of model that sits behind an autocomplete bar or a predictive keyboard. Given the words typed so far, it outputs a probability for every word in the vocabulary, and the app shows the top 5 as suggestions.

**What v2 does differently from v1**

| v1 problem | v2 fix |
|---|---|
| `train_test_split` on n-gram prefixes, so prefixes of the same line ended up in both train and validation (leakage) | Split **by line** *before* building sequences, 80/10/10 train/val/test |
| Val accuracy far below train accuracy (overfitting), and EarlyStopping didn't actually stop training | Smaller effective vocabulary, dropout after both recurrent layers, `EarlyStopping(patience=3, restore_best_weights=True)`, best-checkpoint reload, and the stop epoch is logged |
| One-hot targets over the full vocabulary (memory-heavy) | Integer targets with `sparse_categorical_crossentropy` |
| Only top-1 accuracy, no reference point | Top-1, top-5 and perplexity on a held-out **test** set, compared against two n-gram baselines |

The notebook is self-contained and runs top to bottom on Colab (T4 GPU recommended: *Runtime → Change runtime type → T4 GPU*).

## 1. Setup

Install the dependencies (Colab already ships TensorFlow, so this is mostly a no-op there), fix every random seed to 42 so the line shuffle and weight initialisation can be reproduced, and check whether a GPU is visible. GPU kernels (cuDNN) aren't bit-for-bit deterministic, so a re-run can still differ slightly in the last decimal places.

In [ ]:
%pip install -q "tensorflow>=2.15" nltk numpy pandas matplotlib

In [ ]:
import json
import math
import os
import pickle
import random
import re
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
from matplotlib.ticker import MaxNLocator, PercentFormatter
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "none found, training will run on CPU (slow)")

### Configuration

Every knob in one place. `MAX_TRAIN_SEQUENCES = None` means "use everything"; it exists only so the pipeline can be smoke-tested quickly with a tiny subset.

In [ ]:
# ---- Configuration -------------------------------------------------------
FILE_IDS = ["shakespeare-hamlet.txt", "shakespeare-macbeth.txt", "shakespeare-caesar.txt"]
MIN_TOKENS_PER_LINE = 3        # drop lines shorter than this
SPLIT_FRACTIONS = (0.8, 0.1, 0.1)  # train / val / test, split by line
MIN_WORD_COUNT = 2             # vocabulary = words seen at least this often in train
OOV_TOKEN = "<OOV>"
MAX_LEN_PERCENTILE = 95        # cap the sequence length at this percentile of train lines

EMBED_DIM = 100
LSTM_UNITS = (150, 100)
DROPOUT = 0.3
BATCH_SIZE = 128
EPOCHS = 50
PATIENCE = 3
MAX_TRAIN_SEQUENCES = None     # None = use all training sequences

MODELS_DIR = Path("models")
RESULTS_DIR = Path("results")
FIG_DIR = RESULTS_DIR / "figures"
CKPT_DIR = Path("checkpoints")
for d in (MODELS_DIR, FIG_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Plot styling: two fixed series colours plus recessive axes and grid.
COLORS = {"train": "#2a78d6", "val": "#eb6834",
          "LSTM": "#2a78d6", "GRU": "#eb6834",
          "Unigram baseline": "#1baf7a", "Bigram baseline": "#eda100"}
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "text.color": "#0b0b0b",
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "figure.dpi": 110,
})


def save_fig(fig, name):
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight", dpi=150)
    print("saved", path)

## 2. Data: three Shakespeare plays

We use *Hamlet*, *Macbeth* and *Julius Caesar* from NLTK's Project Gutenberg sample. Three plays instead of one roughly triples the training text while keeping the same style of language.

**Cleaning:**
- remove bracketed text such as the `[The Tragedie of Hamlet ...]` title line
- remove speaker tags at the start of a line (`Ham.`, `Mac.`, `Bru.` ...) and stage-direction lines (`Enter ...`, `Exit`, `Exeunt`, act/scene headings). These aren't dialogue, so they would teach the model formatting rather than language
- lowercase, keep only words (with internal apostrophes such as `'tis` or `o'er`), and collapse whitespace
- keep the play's own line breaks as our "sentences" and drop lines with fewer than 3 tokens, which don't give enough context to learn from

In [ ]:
nltk.download("gutenberg", quiet=True)
from nltk.corpus import gutenberg

BRACKETS = re.compile(r"\[[^\]]*\]|\([^)]*\)")
SPEAKER_TAG = re.compile(r"^\s*[A-Z][A-Za-z]{0,14}\.\s+")   # e.g. "Ham. ", "Mac. "
STAGE_DIRECTION = re.compile(
    r"^\s*(enter|exit|exeunt|manet|actus|act|scena|scoena|scene|flourish|alarum|"
    r"sennet|thunder|hoboyes|drum|trumpets?)\b", re.IGNORECASE)
TOKEN_PATTERN = r"[a-z]+(?:'[a-z]+)*"   # words, keeping internal apostrophes
WORD_RE = re.compile(TOKEN_PATTERN)


def tokenize(text):
    # Lowercase and split into word tokens. The Streamlit app uses the same pattern.
    return WORD_RE.findall(text.lower())


def clean_line(raw):
    line = BRACKETS.sub(" ", raw)
    if STAGE_DIRECTION.match(line):
        return []
    line = SPEAKER_TAG.sub(" ", line)
    return tokenize(line)


lines, line_source = [], []
raw_line_count = 0
for fid in FILE_IDS:
    for raw in gutenberg.raw(fid).split("\n"):
        raw_line_count += 1
        toks = clean_line(raw)
        if len(toks) >= MIN_TOKENS_PER_LINE:
            lines.append(toks)
            line_source.append(fid)

print(f"raw lines: {raw_line_count:,}  ->  kept lines (>= {MIN_TOKENS_PER_LINE} tokens): {len(lines):,}")
for fid in FILE_IDS:
    print(f"  {fid:28s} {line_source.count(fid):,} lines")
print("\nexamples:")
for toks in lines[:8]:
    print("  ", " ".join(toks))

### Exploratory data analysis

Before modelling, check how much text there is, how big the vocabulary is, how long the lines are (this decides the input length), and which words dominate. The most frequent words set the bar that the "always predict the most frequent word" baseline gets for free.

In [ ]:
all_tokens = [t for toks in lines for t in toks]
freq = Counter(all_tokens)
line_lengths = np.array([len(toks) for toks in lines])

eda = {
    "total_lines": len(lines),
    "total_tokens": len(all_tokens),
    "unique_words": len(freq),
    "words_seen_once": sum(1 for c in freq.values() if c == 1),
    "line_len_mean": float(line_lengths.mean()),
    "line_len_median": float(np.median(line_lengths)),
    "line_len_p95": float(np.percentile(line_lengths, 95)),
    "line_len_max": int(line_lengths.max()),
}
display(pd.Series(eda, name="value").to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

ax = axes[0]
bins = np.arange(line_lengths.min(), line_lengths.max() + 2) - 0.5
ax.hist(line_lengths, bins=bins, color=COLORS["train"], edgecolor="#fcfcfb", linewidth=1)
ax.axvline(np.percentile(line_lengths, 95), color="#52514e", linestyle="--", linewidth=1.2)
ax.text(np.percentile(line_lengths, 95) + 0.3, ax.get_ylim()[1] * 0.92, "95th percentile",
        color="#52514e", fontsize=9)
ax.set_title("Line length (tokens)", loc="left")
ax.set_xlabel("tokens per line")
ax.set_ylabel("lines")

ax = axes[1]
top20 = freq.most_common(20)
words, counts = zip(*top20)
ax.barh(range(len(words))[::-1], counts, color=COLORS["train"], height=0.7)
ax.set_yticks(range(len(words))[::-1])
ax.set_yticklabels(words)
ax.set_title("Top-20 most frequent words", loc="left")
ax.set_xlabel("count")
ax.grid(axis="y", visible=False)

fig.tight_layout()
save_fig(fig, "eda_line_lengths_top_words")
plt.show()

## 3. Split by line, before building sequences

Next-word training data is built by turning each line into all of its prefixes:

```
"to be or not to be"  ->  [to] -> be,  [to be] -> or,  [to be or] -> not,  ...
```

v1 created these prefixes first and *then* called `train_test_split` on them. That leaks. The validation example `[to be or] -> not` and the training example `[to be or not] -> to` come from the same line, so the model has already seen the validation context and its answer inside a longer training prefix. Validation then measures memorisation of lines the model has partly seen, not generalisation to new text.

The fix is to treat the **line** as the unit. Shuffle the lines with the fixed seed, split them 80/10/10 into train/validation/test, and only then generate prefixes inside each split. No line contributes to more than one split.

- **train** fits the weights
- **validation** drives early stopping and checkpoint selection
- **test** is touched **once**, at the end, for the final comparison table

In [ ]:
rng = np.random.default_rng(SEED)
order = rng.permutation(len(lines))
n_train = int(SPLIT_FRACTIONS[0] * len(lines))
n_val = int(SPLIT_FRACTIONS[1] * len(lines))

train_lines = [lines[i] for i in order[:n_train]]
val_lines = [lines[i] for i in order[n_train:n_train + n_val]]
test_lines = [lines[i] for i in order[n_train + n_val:]]

for name, split in [("train", train_lines), ("val", val_lines), ("test", test_lines)]:
    print(f"{name:5s} {len(split):6,} lines  {sum(map(len, split)):8,} tokens")

## 4. Tokenizer: fit on train only, with a capped vocabulary

The word → integer mapping is built **from the training lines only**. If validation or test words shaped the vocabulary, information about held-out text would leak into the model's design.

The vocabulary keeps only words that appear **at least twice** in train. Every other word maps to `<OOV>` (out-of-vocabulary). Why cap:
- a word seen once gives the model a single example to learn its embedding and output weights from, which is mostly noise and a major source of overfitting
- the output softmax has one unit per word, so a smaller vocabulary means fewer parameters

Index `0` is reserved for padding and index `1` for `<OOV>`. The tokenizer is stored as a plain dictionary so the Streamlit app can load it without needing any class definition from this notebook.

In [ ]:
train_counts = Counter(t for toks in train_lines for t in toks)
kept_words = sorted((w for w, c in train_counts.items() if c >= MIN_WORD_COUNT),
                    key=lambda w: (-train_counts[w], w))

PAD_ID, OOV_ID = 0, 1
word_index = {OOV_TOKEN: OOV_ID}
for i, w in enumerate(kept_words, start=2):
    word_index[w] = i
index_word = {i: w for w, i in word_index.items()}
VOCAB_SIZE = len(word_index) + 1   # +1 for padding index 0


def encode(tokens):
    return [word_index.get(t, OOV_ID) for t in tokens]


def oov_rate(split_lines):
    toks = [t for l in split_lines for t in l]
    return sum(t not in word_index for t in toks) / len(toks)


oov = {name: oov_rate(s) for name, s in
       [("train", train_lines), ("val", val_lines), ("test", test_lines)]}
print(f"distinct words in train: {len(train_counts):,}")
print(f"vocabulary (count >= {MIN_WORD_COUNT}): {len(kept_words):,} words  ->  VOCAB_SIZE = {VOCAB_SIZE:,} (incl. pad + OOV)")
for k, v in oov.items():
    print(f"OOV share of {k:5s} tokens: {v:.2%}")

## 5. Build n-gram sequences

For each line, every prefix becomes an input and the following word becomes the target.

- **Max length.** A few very long lines would force every sequence to be padded to their length. Instead we take the 95th percentile of train line lengths as the cap; longer contexts are **truncated from the left**, keeping the words closest to the one being predicted.
- **Pre-padding.** Zeros go at the *start* so the real words sit at the end, right next to the LSTM's final state that makes the prediction.
- **Integer targets.** The target is the word's index (a single int), not a one-hot vector of length `VOCAB_SIZE`. With `sparse_categorical_crossentropy` this gives exactly the same loss while storing about `VOCAB_SIZE` times less target data.
- **OOV targets are dropped.** A sample whose answer is `<OOV>` would teach the model to suggest "unknown word", which is useless for autocomplete and would inflate accuracy. Their share is reported below. OOV words are still allowed in the *context*.

In [ ]:
MAX_SEQ_LEN = int(np.percentile([len(l) for l in train_lines], MAX_LEN_PERCENTILE))
INPUT_LEN = MAX_SEQ_LEN - 1   # context length fed to the model
print(f"max sequence length ({MAX_LEN_PERCENTILE}th pct of train lines): {MAX_SEQ_LEN}  ->  model input length {INPUT_LEN}")


def make_sequences(split_lines):
    X, y, dropped = [], [], 0
    for toks in split_lines:
        ids = encode(toks)
        for i in range(1, len(ids)):
            if ids[i] == OOV_ID:
                dropped += 1
                continue
            context = ids[max(0, i - INPUT_LEN):i]          # left-truncate
            X.append([PAD_ID] * (INPUT_LEN - len(context)) + context)  # pre-pad
            y.append(ids[i])
    return np.array(X, dtype=np.int32), np.array(y, dtype=np.int32), dropped


X_train, y_train, drop_tr = make_sequences(train_lines)
X_val, y_val, drop_va = make_sequences(val_lines)
X_test, y_test, drop_te = make_sequences(test_lines)

for name, X, d in [("train", X_train, drop_tr), ("val", X_val, drop_va), ("test", X_test, drop_te)]:
    print(f"{name:5s} sequences: {len(X):7,}   (dropped {d:,} with OOV target, {d / (len(X) + d):.1%})")

if MAX_TRAIN_SEQUENCES is not None:   # smoke-test mode only
    idx = np.random.default_rng(SEED).permutation(len(X_train))[:MAX_TRAIN_SEQUENCES]
    X_train, y_train = X_train[idx], y_train[idx]
    print(f"\n** SMOKE TEST: training on {len(X_train):,} sequences only **")

print("\nexample:", [index_word.get(i, "<pad>") for i in X_train[0]], "->", index_word[y_train[0]])

## 6. Baselines (evaluated on test)

A neural model's number means little without a reference point. Two cheap baselines:

- **(a) Unigram / most-frequent word.** Always predict the most frequent target word in train (top-5 = the 5 most frequent). Its accuracy is the floor any model must beat.
- **(b) Bigram counts.** Look only at the *previous* word and predict whatever most often followed it in train. Its top-5 is the 5 most frequent followers, padded with the unigram list when a word had fewer than 5 followers. If the previous word was never seen, it falls back to (a).

To compute **perplexity** a baseline must assign a probability to every word:
- the unigram model uses add-one (Laplace) smoothed counts
- the bigram model interpolates, `P = λ·P_bigram + (1-λ)·P_unigram`, with `λ` picked on the **validation** set, never on test

**Perplexity** is `exp(mean cross-entropy)`. Intuitively it's the number of words the model is "as confused as if it were choosing uniformly among". Lower is better, and a perfect model scores 1.

In [ ]:
prev_train = X_train[:, -1]   # last context token (always a real word, never padding)
prev_val, prev_test = X_val[:, -1], X_test[:, -1]

# (a) unigram over targets (index 0 = pad and 1 = OOV are never targets)
uni_counts = np.bincount(y_train, minlength=VOCAB_SIZE).astype(np.float64)
uni_counts[[PAD_ID, OOV_ID]] = 0
n_classes = VOCAB_SIZE - 2
uni_probs = (uni_counts + 1) / (uni_counts.sum() + n_classes)
uni_probs[[PAD_ID, OOV_ID]] = 0
uni_top5 = np.argsort(-uni_counts)[:5]

# (b) bigram counts: previous word -> Counter(next word)
bigram = defaultdict(Counter)
for p, t in zip(prev_train, y_train):
    bigram[int(p)][int(t)] += 1


def bigram_topk(p, k=5):
    picks = [w for w, _ in bigram[p].most_common(k)] if p in bigram else []
    for w in uni_top5:
        if len(picks) >= k:
            break
        if w not in picks:
            picks.append(int(w))
    return picks


def bigram_prob(p, t, lam):
    pu = uni_probs[t]
    if p not in bigram:
        return pu
    c = bigram[p]
    return lam * c[t] / sum(c.values()) + (1 - lam) * pu


def evaluate_unigram(y):
    top1 = np.mean(y == uni_top5[0])
    top5 = np.mean(np.isin(y, uni_top5))
    ppl = math.exp(-np.mean(np.log(uni_probs[y])))
    return top1, top5, ppl


def evaluate_bigram(prev, y, lam):
    preds = [bigram_topk(int(p)) for p in prev]
    top1 = np.mean([pr[0] == t for pr, t in zip(preds, y)])
    top5 = np.mean([t in pr for pr, t in zip(preds, y)])
    ppl = math.exp(-np.mean([math.log(bigram_prob(int(p), int(t), lam)) for p, t in zip(prev, y)]))
    return top1, top5, ppl


# choose the interpolation weight on validation
lam_grid = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
val_ppl = {lam: evaluate_bigram(prev_val, y_val, lam)[2] for lam in lam_grid}
BEST_LAMBDA = min(val_ppl, key=val_ppl.get)
print("bigram val perplexity by lambda:", {k: round(v, 1) for k, v in val_ppl.items()})
print("chosen lambda:", BEST_LAMBDA)

baseline_results = {}
for name, (t1, t5, ppl) in [
    ("Unigram baseline", evaluate_unigram(y_test)),
    ("Bigram baseline", evaluate_bigram(prev_test, y_test, BEST_LAMBDA)),
]:
    baseline_results[name] = {"test_top1_accuracy": float(t1), "test_top5_accuracy": float(t5),
                              "test_perplexity": float(ppl)}
pd.DataFrame(baseline_results).T

## 7. Models: stacked LSTM vs stacked GRU

Both models share one architecture and differ only in the recurrent cell:

```
Embedding(VOCAB_SIZE, 100) -> RNN(150, return_sequences=True) -> Dropout(0.3)
                           -> RNN(100) -> Dropout(0.3) -> Dense(VOCAB_SIZE, softmax)
```

**Embedding.** Each word index is looked up in a trainable 100-dimensional vector table. Words used in similar contexts (`lord`/`king`, `thee`/`thou`) end up with similar vectors, so what the model learns about one word carries over to its neighbours. That isn't possible with sparse one-hot inputs.

**LSTM gates.** An LSTM keeps a separate *cell state* that runs through time, controlled by three gates:
- the *forget* gate decides what to erase
- the *input* gate decides what new information to write
- the *output* gate decides what to expose as the hidden state

The additive cell-state path lets gradients flow over many steps without vanishing.

**GRU gates.** A GRU merges the cell and hidden state and uses only two gates:
- an *update* gate (a combined forget/input gate)
- a *reset* gate (how much of the past to use when proposing a new state)

That's about 25% fewer parameters per unit, often similar accuracy, and faster training. Comparing the two on identical data is the point of this experiment.

**Stacking.** The first recurrent layer returns its full output sequence so the second can read it, which gives a hierarchy of features over the context.

**Dropout (0.3).** During training, 30% of activations are randomly zeroed after each recurrent layer. The network can't rely on any single unit, which reduces the memorisation behind v1's train/val gap.

**Sparse categorical cross-entropy.** The loss is `-log p(correct word)`, read directly from the integer target. It's identical to categorical cross-entropy with one-hot labels, without building the one-hot matrix.

**Metrics.**
- accuracy (top-1)
- `SparseTopKCategoricalAccuracy(k=5)`: whether the true word is among the 5 highest-probability words. An autocomplete bar shows several suggestions, so top-5 is the metric that matches the product.

**Callbacks.**
- `EarlyStopping(val_loss, patience=3, restore_best_weights=True)` halts once validation loss hasn't improved for 3 epochs
- `ModelCheckpoint` saves the weights from the epoch with the lowest validation loss; we reload that file after training so the evaluated model is the best one, however training ended

In [ ]:
def build_model(cell, name):
    rnn = {"LSTM": layers.LSTM, "GRU": layers.GRU}[cell]
    model = keras.Sequential([
        keras.Input(shape=(INPUT_LEN,), dtype="int32"),
        layers.Embedding(VOCAB_SIZE, EMBED_DIM),
        rnn(LSTM_UNITS[0], return_sequences=True),
        layers.Dropout(DROPOUT),
        rnn(LSTM_UNITS[1]),
        layers.Dropout(DROPOUT),
        layers.Dense(VOCAB_SIZE, activation="softmax"),
    ], name=name)
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=keras.optimizers.Adam(),
        metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_accuracy")],
    )
    return model


build_model("LSTM", "lstm").summary()
build_model("GRU", "gru").summary()

### Training

Each model trains for up to 50 epochs with early stopping on validation loss. We record:
- wall-clock training time
- the epoch where training actually stopped
- the epoch whose weights were kept (the best validation loss)

In [ ]:
def train(cell):
    tf.keras.utils.set_random_seed(SEED)   # identical init/shuffle conditions for both models
    model = build_model(cell, cell.lower())
    ckpt_path = CKPT_DIR / f"{cell.lower()}_best.keras"
    early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                               restore_best_weights=True, verbose=1)
    checkpoint = keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss",
                                                 save_best_only=True, verbose=0)
    start = time.time()
    history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                        epochs=EPOCHS, batch_size=BATCH_SIZE,
                        callbacks=[early_stop, checkpoint], verbose=2)
    train_time = time.time() - start

    epochs_run = len(history.history["loss"])
    best_epoch = int(np.argmin(history.history["val_loss"])) + 1
    stopped_early = early_stop.stopped_epoch > 0
    print(f"\n[{cell}] trained {epochs_run} epoch(s) in {train_time:.1f}s; "
          f"{'early-stopped' if stopped_early else 'reached max epochs'}; "
          f"best val_loss at epoch {best_epoch}")

    best_model = keras.models.load_model(ckpt_path)   # guaranteed best-val-loss weights
    return {"model": best_model, "history": history.history, "train_time_s": train_time,
            "epochs_run": epochs_run, "best_epoch": best_epoch, "stopped_early": stopped_early,
            "best_val_loss": float(min(history.history["val_loss"])),
            "params": int(best_model.count_params()), "ckpt_path": ckpt_path}


runs = {}
runs["LSTM"] = train("LSTM")

In [ ]:
runs["GRU"] = train("GRU")

### Training curves

Train vs validation loss and top-5 accuracy per epoch. A widening gap, with train still improving while validation flattens or worsens, is the signature of overfitting. The dashed line marks the epoch whose weights were kept.

In [ ]:
for cell, run in runs.items():
    h = run["history"]
    ep = np.arange(1, len(h["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for ax, key, title in [(axes[0], "loss", "Loss (sparse cross-entropy)"),
                           (axes[1], "top5_accuracy", "Top-5 accuracy")]:
        ax.plot(ep, h[key], color=COLORS["train"], marker="o", markersize=4, label="train")
        ax.plot(ep, h[f"val_{key}"], color=COLORS["val"], marker="o", markersize=4, label="validation")
        ax.axvline(run["best_epoch"], color="#52514e", linestyle="--", linewidth=1)
        ax.set_title(f"{cell}: {title}", loc="left")
        ax.set_xlabel("epoch")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        if key == "top5_accuracy":
            ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        ax.legend(frameon=False)
    fig.tight_layout()
    save_fig(fig, f"training_curves_{cell.lower()}")
    plt.show()

## 8. Final evaluation on the test set

Up to here the test set has only been used by the baselines, which aren't tuned on it. Now each model's best checkpoint is evaluated on it **once**. Model perplexity is `exp(test loss)`, since the Keras loss is the mean cross-entropy.

In [ ]:
results = dict(baseline_results)
for cell, run in runs.items():
    loss, acc, top5 = run["model"].evaluate(X_test, y_test, batch_size=512, verbose=0)
    run["test"] = {"test_loss": float(loss), "test_top1_accuracy": float(acc),
                   "test_top5_accuracy": float(top5), "test_perplexity": float(math.exp(loss))}
    results[cell] = {k: v for k, v in run["test"].items() if k != "test_loss"}
    results[cell].update({"params": run["params"], "train_time_s": run["train_time_s"],
                          "epochs_run": run["epochs_run"], "best_epoch": run["best_epoch"]})

table = pd.DataFrame(results).T[["test_top1_accuracy", "test_top5_accuracy", "test_perplexity",
                                 "params", "train_time_s", "epochs_run", "best_epoch"]]
table = table.loc[["Unigram baseline", "Bigram baseline", "LSTM", "GRU"]]
table.style.format({"test_top1_accuracy": "{:.2%}", "test_top5_accuracy": "{:.2%}",
                    "test_perplexity": "{:.1f}", "params": "{:,.0f}",
                    "train_time_s": "{:.1f}", "epochs_run": "{:.0f}", "best_epoch": "{:.0f}"},
                   na_rep="–")

In [ ]:
order_names = ["Unigram baseline", "Bigram baseline", "LSTM", "GRU"]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, key, title, fmt in [
    (axes[0], "test_top1_accuracy", "Top-1 accuracy (higher is better)", "{:.1%}"),
    (axes[1], "test_top5_accuracy", "Top-5 accuracy (higher is better)", "{:.1%}"),
    (axes[2], "test_perplexity", "Perplexity (lower is better)", "{:.0f}"),
]:
    vals = [results[n][key] for n in order_names]
    bars = ax.bar(order_names, vals, color=[COLORS[n] for n in order_names], width=0.6)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), fmt.format(v),
                ha="center", va="bottom", fontsize=9, color="#0b0b0b")
    ax.set_title(title, loc="left")
    ax.margins(y=0.15)
    if "accuracy" in key:
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.tick_params(axis="x", labelrotation=20)
    ax.grid(axis="x", visible=False)
fig.tight_layout()
save_fig(fig, "test_comparison")
plt.show()

## 9. Qualitative check

Metrics summarise; examples show what a user would actually see. Below are the top-5 suggestions (with probabilities) from the selected model for a handful of prompts. They're shown as they come out, good or bad. Prompts are tokenised exactly like the training data, unknown words become `<OOV>`, and `<OOV>`/padding are never suggested.

In [ ]:
best_cell = min(runs, key=lambda c: runs[c]["best_val_loss"])   # chosen on validation, not test
best_model = runs[best_cell]["model"]
print(f"selected model (lowest validation loss): {best_cell}")


def top_k_next_words(model, text, k=5):
    ids = encode(tokenize(text))[-INPUT_LEN:]
    x = np.array([[PAD_ID] * (INPUT_LEN - len(ids)) + ids], dtype=np.int32)
    probs = model.predict(x, verbose=0)[0].copy()
    probs[[PAD_ID, OOV_ID]] = 0
    top = np.argsort(-probs)[:k]
    return [(index_word[i], float(probs[i])) for i in top]


PROMPTS = ["to be or not to", "my lord", "the king", "what is", "i will",
           "good night", "o my", "let us"]
rows = []
for p in PROMPTS:
    preds = top_k_next_words(best_model, p)
    rows.append({"prompt": p, **{f"#{i + 1}": f"{w} ({pr:.1%})" for i, (w, pr) in enumerate(preds)}})
pd.DataFrame(rows).set_index("prompt")

## 10. Save artefacts

- `models/next_word_model.keras`: the model with the lowest **validation** loss (selection never looks at test)
- `models/tokenizer.pickle`: a plain dict with `word_index`, `index_word`, the OOV token and the token regex, so the app tokenises exactly like this notebook
- `results/metrics.json`: all test metrics, vocabulary size, sequence length, stop epochs, parameter counts and timings
- `results/figures/*.png`: every plot above

Everything is zipped and, on Colab, downloaded automatically. Unzip it into the repository root so the Streamlit app can find `models/` and `results/`.

In [ ]:
best_model.save(MODELS_DIR / "next_word_model.keras")
tokenizer_payload = {
    "word_index": word_index,
    "index_word": index_word,
    "oov_token": OOV_TOKEN,
    "pad_id": PAD_ID,
    "oov_id": OOV_ID,
    "token_pattern": TOKEN_PATTERN,
    "max_sequence_len": MAX_SEQ_LEN,
}
with open(MODELS_DIR / "tokenizer.pickle", "wb") as f:
    pickle.dump(tokenizer_payload, f, protocol=pickle.HIGHEST_PROTOCOL)

metrics = {
    "selected_model": best_cell,
    "selection_criterion": "lowest validation loss",
    "seed": SEED,
    "smoke_test": MAX_TRAIN_SEQUENCES is not None,
    "data": {
        "files": FILE_IDS,
        **eda,
        "split_lines": {"train": len(train_lines), "val": len(val_lines), "test": len(test_lines)},
        "sequences": {"train": int(len(X_train)), "val": int(len(X_val)), "test": int(len(X_test))},
        "oov_token_share": oov,
    },
    "vocab_size": VOCAB_SIZE,
    "min_word_count": MIN_WORD_COUNT,
    "max_sequence_len": MAX_SEQ_LEN,
    "input_len": INPUT_LEN,
    "bigram_lambda": BEST_LAMBDA,
    "results": results,
    "training": {cell: {"epochs_run": r["epochs_run"], "best_epoch": r["best_epoch"],
                        "stopped_early": bool(r["stopped_early"]), "best_val_loss": r["best_val_loss"],
                        "params": r["params"], "train_time_s": r["train_time_s"]}
                 for cell, r in runs.items()},
    "config": {"embed_dim": EMBED_DIM, "rnn_units": list(LSTM_UNITS), "dropout": DROPOUT,
               "batch_size": BATCH_SIZE, "max_epochs": EPOCHS, "patience": PATIENCE},
}
with open(RESULTS_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics["results"], indent=2))

In [ ]:
# Build the zip with models/ and results/ at its top level.
archive = "next_word_prediction_artifacts.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in (MODELS_DIR, RESULTS_DIR):
        for path in folder.rglob("*"):
            if path.is_file():
                zf.write(path, path.as_posix())
print("created", archive, f"({os.path.getsize(archive) / 1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not running on Colab; the zip is in the working directory.")

## 11. Conclusions and limitations

### Conclusions
Based on the full run on Colab (T4 GPU, seed 42). Numbers are on the held-out test set (4,801 predictions) and match `results/metrics.json`. A re-run can differ slightly in the last decimals because GPU kernels aren't fully deterministic.

| Model | Top-1 | Top-5 | Perplexity |
|---|---:|---:|---:|
| Unigram baseline | 3.35% | 12.87% | 454.4 |
| Bigram baseline | 9.39% | 23.89% | 284.3 |
| LSTM | 8.31% | 22.27% | 274.4 |
| GRU (selected) | 9.29% | 23.89% | 259.8 |

- **Did the neural models beat the baselines?** Both clearly beat the unigram floor: top-5 nearly doubles and perplexity falls by about 40%. Against the bigram count model the margin is small. The GRU **ties it on top-5** (23.89%), is **0.1 points lower on top-1**, and is clearly better only on **perplexity** (259.8 vs 284.3). The LSTM is below the bigram baseline on both accuracies. So the networks spread probability more sensibly over the vocabulary, but on this small corpus that rarely changes *which* words reach the top 5. Most of the learnable signal here is local, one or two words back.
- **LSTM vs GRU:** the GRU wins on every metric (top-1 9.29% vs 8.31%, top-5 23.89% vs 22.27%, perplexity 259.8 vs 274.4). It also has about 7.5% fewer parameters (766,272 vs 828,272) and trained about 21% faster (45.5 s vs 57.6 s). It also had the lower validation loss (5.55 vs 5.60), which is why it was selected and deployed. With about 40k training sequences the simpler cell is the better fit.
- **Overfitting:** early stopping worked as intended. The LSTM stopped after 15 epochs (best at 12) and the GRU after 13 (best at 10), out of a maximum of 50. At the kept epoch, train and validation top-5 accuracy are within about 1–2 points, so v1's large train/val gap is gone. After that point training loss keeps falling while validation loss flattens, which is the onset of overfitting that the callback cuts off.
- **Qualitative predictions:** the model has learned frequent local patterns: `o my` → *lord* (42.0%), `i will` → *not* (19.2%) / *be*, `the king` → *is* / *of*, `what is` → *a* / *the*. It hasn't memorised famous lines. For `to be or not to`, *be* is only the 2nd guess (3.3%) behind *the*. Because of the line-level split the model may never have seen that line, and the corpus is too small to learn quotations from elsewhere. Archaic spellings (`heare`, `haue`, `selfe`, `deere`) show up in the suggestions as they appear in the source text.
- **Takeaway:** v2's numbers are lower than v1's headline figures but trustworthy. There's no leakage, the test set was used once, and baselines give context. Beating a bigram model convincingly will need more data and/or subword tokenisation and longer context (see below).

### Limitations
- **Small corpus.** Three plays is tens of thousands of tokens; modern language models train on billions. Expect absolute accuracies to stay modest.
- **Archaic spelling.** The Gutenberg texts use Elizabethan, inconsistent spelling (`haue`, `vpon`, `selfe`), so the same word can be split across several vocabulary entries, and the model won't transfer well to modern English.
- **Word-level vocabulary.** Any word outside the capped vocabulary becomes `<OOV>` and can never be suggested. Rare words and new names are invisible to the model.
- **No subword tokenisation.** BPE/WordPiece would share statistics across `king`/`kings`/`kingdom` and remove the OOV problem.
- **Short context.** Inputs are single lines, capped at the 95th-percentile length, so context from previous lines is lost.
- **Next step: a transformer.** A small decoder-only transformer with subword tokenisation (or fine-tuning a pretrained one such as GPT-2) would capture longer context and is the natural v3.